In [5]:
"""
b2_palatal.py — B2: palatalized consonants (tʲ, kʲ …) where the ʲ is INSIDE the consonant,
not an extra [j] after it. Every occurrence of the chosen consonants in the text is changed.

Put this file next to functions.py, accent_tools.py and b4_accent.py (it reuses b4_accent's player page).
Edit the SETTINGS and run   python b2_palatal.py
or in a notebook:           from b2_palatal import run;  run(weights=[0.4, 0.7, 1.0])

WHY STOCK KOKORO ADDS A [j]
  "tʲ" is two tokens, t and ʲ. Each gets its own frames (at least one 25 ms frame), so the model
  plays t, then ʲ — which sounds like t + j.

WHAT THIS SCRIPT DOES (per sentence)
  1. A = original IPA, B = the same IPA with every t -> tʲ (B has one extra token per change).
  2. Durations: the ʲ gets ZERO frames.
       TIMING="original": every token keeps the duration it has in the ORIGINAL text, so the
                          sentence is exactly as long as the original, and t is as long as a plain t
       TIMING="merged"  : B's own durations, the ʲ frames are added to the consonant
  3. Soft alignment (B2) — inside the consonant's frames the ʲ vector is mixed in:
       ʲ weight rises from RAMP_START·w to w across the consonant (palatalization strongest at
       the release), then fades from w to 0 over the next SPILL_FRAMES frames (the vowel onset,
       where palatalization shows up as a high F2 transition). Every frame still sums to 1.
     The same soft alignment drives pitch/energy (step 8) and the acoustic frames (step 11).
  4. The encoders read B (with ʲ), so the neighbours are also coloured by the ʲ (context effect).

WHY IT CAN STILL SOUND LIKE t + j, AND THE SWITCHES THAT TEST EACH CAUSE
  CONTEXT     the encoders read "tʲa", and the vowel's own vector learned a [j] on-glide from training
              (Japanese kʲa = [kja]). Even with 0 frames for ʲ, the vowel starts with a glide.
              "original" -> every token except the consonant keeps its vector from the ORIGINAL text.
  CONS_VECTOR the consonant's own vector: "palatal" = t as encoded before ʲ (palatal colouring
              learned by the model) | "plain" = t from the original text.
  PROSODY     "original" -> pitch/energy from the original text (t stays voiceless; a voiced ʲ blend
              in pitch/energy makes the t partly voiced = j-like) | "blended" -> from the soft alignment.
  SPILL_FRAMES fading ʲ into the vowel = literally a glide; keep it at 0-1 frame.
  VARIANTS below renders several combinations into the same page so you can hear which cause matters.

PAGE (OUT_DIR/b2_<consonants>_palatal.html): one player per condition
  original                 the text without changes
  stock tʲ (extra j)       what Kokoro does by itself: t and ʲ with their own frames
  B2 w=…                   one player per weight; dashed pitch curves = same timing with ʲ weight 0
"""
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

import accent_tools as AT
from accent_tools import load_story, manipulate_with_quota, make_alignment, block_changes, voiced_mask
from functions import model, VOCAB, BOUNDARY, SR, align_phonemes, _encode, _voice_pack
from b4_accent import _player_fragment, _page, _name

# =====================================================================================================
# SETTINGS
# =====================================================================================================
TEXT = """La casa di Carlo è dietro il parco; Il cane corre sul campo.
Chiara compra un chilo di pane; Questo libro costa poco.""" 

VOICE = "if_sara"
CONSONANTS = ["k"]            # every occurrence of these gets a ʲ, e.g. ["t"], ["k"], ["t", "k"]

WEIGHTS = [0.4, 0.7]          # peak weight of ʲ inside the consonant (one number or a list)
RAMP_START = 0.3              # ʲ weight at the START of the consonant = RAMP_START · w (0 = starts plain)
SPILL_FRAMES = 1              # frames (25 ms each) after the consonant where the ʲ fades out (0 = none)
TIMING = "original"           # "original": same timing as the original text | "merged": B's own timing
CONTEXT = "original"          # "original": neighbours keep their ORIGINAL vectors | "palatal": as encoded with ʲ
CONS_VECTOR = "palatal"       # consonant vector: "palatal" (t as encoded before ʲ) | "plain" (original t)
PROSODY = "original"          # "original": pitch/energy of the original text | "blended": from the soft alignment

# Several combinations on one page (each overrides the settings above; "weights" optional).
# Set VARIANTS = None to render only the settings above.
VARIANTS = [
    dict(label="ʲ inside t, original context + prosody"),
    dict(label="palatal t vector only, no ʲ frames (B4-style)", weights=[0.0]),
    dict(label="ʲ inside t, palatal context (vowel learned the glide)", CONTEXT="palatal"),
    dict(label="previous version (palatal context, blended prosody, 2-frame fade)",
         CONTEXT="palatal", PROSODY="blended", SPILL_FRAMES=2),
]

MAKE_REFERENCE = True         # include the original text
MAKE_STOCK = True             # include stock Kokoro tʲ (the "extra j" version) for comparison
NORMALIZE = None              # load_story IPA clean-up (None = its default ɪ->i; {} = off)
SPEED = 1.0
SEED = 0
OUT_DIR = "b2_output"
PAL = "ʲ"


# =====================================================================================================
# CORE
# =====================================================================================================
@torch.no_grad()
def _prosody(enc, s, speed):
    """Steps 5-6: prosody features d and predicted durations (list incl. pads)."""
    d = model.predictor.text_encoder(enc["d_en"], s, enc["L"], enc["tm"])
    x, _ = model.predictor.lstm(d)
    dur = torch.round(torch.sigmoid(model.predictor.duration_proj(x)).sum(-1) / speed).clamp(min=1).long()
    return d, dur.reshape(-1).tolist()


def _kept(ps):
    return [j for j, c in enumerate(ps) if c in VOCAB]


def _structure(psA, psB, maskB):
    """Token bookkeeping between A and B.
    pairs     : [(consonant token, ʲ token)] in B
    b_from_a  : for every B token, the A token it corresponds to (None for an inserted ʲ)
    altered_A : per A character, True if that consonant got a ʲ"""
    keptB = _kept(psB)
    tokB = {j: k + 1 for k, j in enumerate(keptB)}
    inserted = {j for j in keptB if psB[j] == PAL and maskB[j]}
    pairs = [(tokB[j] - 1, tokB[j]) for j in sorted(inserted)]
    b_from_a, a_tok = [0], 1
    for j in keptB:
        if j in inserted:
            b_from_a.append(None)
        else:
            b_from_a.append(a_tok); a_tok += 1
    b_from_a.append(a_tok)                                        # end pad
    assert a_tok == len(_kept(psA)) + 1, "B is not 'A with ʲ inserted' — check CONSONANTS / the IPA"
    altered_A, ia = [False] * len(psA), 0                         # walk the characters of B and A together
    for j, c in enumerate(psB):
        if j in inserted:
            continue
        if maskB[j]:
            altered_A[ia] = True
        ia += 1
    return pairs, b_from_a, altered_A


def soft_alignment(dur, pairs, w, ramp_start=0.3, spill=2):
    """Hard alignment from dur (ʲ tokens have 0 frames), then for every (c, j) pair:
    consonant frames: ʲ weight ramps ramp_start·w -> w, consonant = 1 - that;
    next `spill` frames: ʲ weight fades w -> 0, whatever token owns them is scaled by 1 - that."""
    n = len(dur)
    ends = np.cumsum(dur); starts = ends - np.asarray(dur)
    T = int(ends[-1])
    aln = torch.zeros(n, T, device=model.device)
    for t_, (a, b) in enumerate(zip(starts, ends)):
        aln[t_, int(a):int(b)] = 1
    for c, j in pairs:
        cs, ce = int(starts[c]), int(ends[c])
        if ce > cs and w > 0:
            r = torch.linspace(ramp_start * w, w, ce - cs, device=aln.device)
            aln[:, cs:ce] = 0
            aln[c, cs:ce] = 1 - r
            aln[j, cs:ce] = r
        k = min(spill, T - ce)
        if k > 0 and w > 0:
            r2 = torch.linspace(w, 0, k + 1, device=aln.device)[:k]
            aln[:, ce:ce + k] *= (1 - r2)
            aln[j, ce:ce + k] += r2
    return aln.unsqueeze(0)


@torch.no_grad()
def render_sentence(psA, psB, maskB, pack, w, timing="original", ramp_start=0.3, spill=1,
                    speed=1.0, seed=0, stock=False, context="original", cons_vector="palatal",
                    prosody="original"):
    """One sentence. Returns audio, display string, display durations, display mask, F0, N, F0_ref, N_ref."""
    refB = pack[len(psB) - 1].to(model.device)
    eB = _encode(psB)
    dB, durB = _prosody(eB, refB[:, 128:], speed)
    f = lambda z: z.squeeze().cpu().numpy()

    if stock:                                                     # Kokoro's own t + ʲ (extra j)
        aln = make_alignment(psB, durB)
        F0, N = model.predictor.F0Ntrain(dB.transpose(-1, -2) @ aln, refB[:, 128:])
        torch.manual_seed(seed)
        a = model.decoder(eB["t_en"] @ aln, F0, N, refB[:, :128]).squeeze().cpu().numpy()
        return a, psB, durB, list(maskB), f(F0), f(N), f(F0), f(N)

    pairs, b_from_a, altered_A = _structure(psA, psB, maskB)
    refA = pack[len(psA) - 1].to(model.device)                    # original length -> original voice row
    eA = _encode(psA)
    dA, durA = _prosody(eA, refA[:, 128:], speed)

    if timing == "original":
        dur = [0 if i is None else durA[i] for i in b_from_a]
    elif timing == "merged":
        dur = list(durB)
        for c, j in pairs:
            dur[c] += dur[j]; dur[j] = 0
    else:
        raise ValueError(f"TIMING must be 'original' or 'merged', not {timing!r}")
    disp = [0] * (len(_kept(psA)) + 2)                            # the same frames per ORIGINAL token
    for kb, ka in enumerate(b_from_a):
        if ka is not None:
            disp[ka] += dur[kb]

    # step-10 vectors on B's token positions: neighbours from A or B, consonant from A or B, ʲ from B
    cons = {c for c, _ in pairs}
    t_en = eB["t_en"].clone()
    for kb, ka in enumerate(b_from_a):
        if ka is None:
            continue                                              # ʲ: B's vector
        use_b = (cons_vector == "palatal") if kb in cons else (context == "palatal")
        if not use_b:
            t_en[..., kb] = eA["t_en"][..., ka]

    aln = soft_alignment(dur, pairs, w, ramp_start, spill)
    alnA = make_alignment(psA, disp)                              # original tokens, same frames
    F0r, Nr = model.predictor.F0Ntrain(dA.transpose(-1, -2) @ alnA, refA[:, 128:])
    if prosody == "original":
        F0, N = F0r, Nr
    elif prosody == "blended":
        F0, N = model.predictor.F0Ntrain(dB.transpose(-1, -2) @ aln, refB[:, 128:])
    else:
        raise ValueError(f"PROSODY must be 'original' or 'blended', not {prosody!r}")
    assert aln.shape[-1] == alnA.shape[-1]
    torch.manual_seed(seed)
    a = model.decoder(t_en @ aln, F0, N, refA[:, :128]).squeeze().cpu().numpy()
    return a, psA, disp, altered_A, f(F0), f(N), f(F0r), f(Nr)


# =====================================================================================================
# TEXT -> PAGE
# =====================================================================================================
def _render_text(ipa, manip, masks, pack, w, st, stock=False, original=False):
    segments, seg_words, seg_altered = [], [], []
    T, F0o, F0r, Ne, Nr, sents, audios = [], [], [], [], [], [], []
    t0, word_off = 0.0, 0
    for ci, (psA, psB, mB) in enumerate(zip(ipa, manip, masks)):
        if not psA.strip():
            continue
        if original:                                              # plain original text: B = A, nothing inserted
            psB, mB = psA, [False] * len(psA)
            stock = True
        a, ps_d, dur_d, m_d, f0, n, f0r, nr = render_sentence(
            psA, psB, mB, pack, w, st["timing"], st["ramp_start"], st["spill"], st["speed"],
            st["seed"] + ci, stock=stock, context=st["context"], cons_vector=st["cons_vector"],
            prosody=st["prosody"])
        segs, widx, altd = align_phonemes(ps_d, dur_d, n_samples=len(a), sr=SR,
                                          altered_mask=None if original else m_d)
        segments += [(lab, s0 + t0, s1 + t0, info) for lab, s0, s1, info in segs]
        seg_words += [None if x is None else x + word_off for x in widx]
        seg_altered += altd
        real = [x for x in widx if x is not None]
        word_off += (max(real) + 1) if real else 0
        k = len(f0)
        T += [t0 + np.arange(k) * (len(a) / k) / SR, np.array([t0 + len(a) / SR])]
        F0o += [np.where(voiced_mask(f0), f0, np.nan), np.array([np.nan])]
        F0r += [np.where(voiced_mask(f0r), f0r, np.nan), np.array([np.nan])]
        Ne += [n, np.array([np.nan])]; Nr += [nr, np.array([np.nan])]
        sents.append(dict(chunk=ci, start=t0, end=t0 + len(a) / SR, donor=None, donor_idx=0))
        audios.append(a); t0 += len(a) / SR
    label = "original" if original else ("stock" if stock else f"B2 w={w:g}")
    pros = dict(t=np.concatenate(T), f0=np.concatenate(F0o), f0_t=np.concatenate(F0r),
                n=np.concatenate(Ne), n_t=np.concatenate(Nr), chunks=[], sentences=sents,
                target=st["voice_name"], donors=[label], what=["B2"],
                modes=dict(f0="-", energy="-", dur="-", dur_scope="-", scope="-", assign="-",
                           seed=st["seed"], donor_text="-"))
    return dict(block="all", voice=st["voice_name"], ref_voice=None, audio=np.concatenate(audios),
                segments=segments, seg_words=seg_words, seg_altered=seg_altered, pros=pros)


def run(text=TEXT, voice=VOICE, consonants=CONSONANTS, weights=WEIGHTS, ramp_start=RAMP_START,
        spill_frames=SPILL_FRAMES, timing=TIMING, context=CONTEXT, cons_vector=CONS_VECTOR, prosody=PROSODY,
        variants=VARIANTS, make_reference=MAKE_REFERENCE, make_stock=MAKE_STOCK,
        normalize=NORMALIZE, speed=SPEED, seed=SEED, out_dir=OUT_DIR):
    from html import escape
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    consonants = [consonants] if isinstance(consonants, str) else list(consonants)
    as_list = lambda x: [float(v) for v in (x if isinstance(x, (list, tuple)) else [x])]
    base = dict(WEIGHTS=weights, RAMP_START=ramp_start, SPILL_FRAMES=spill_frames, TIMING=timing,
                CONTEXT=context, CONS_VECTOR=cons_vector, PROSODY=prosody)
    variants = variants or [dict(label="settings")]
    pack, _ = _voice_pack(voice)

    story = load_story(text, name="b2", normalize=normalize, verbose=False)
    ipa = [s_ for b in story["blocks"] for s_ in b["ipa"]]
    words = [w_ for b in story["blocks"] for s_ in b["spelling"] for w_ in s_]
    rule = {"kind": "substitute", "map": {c: c + PAL for c in consonants}}
    manip, masks, origins = manipulate_with_quota(ipa, [rule], return_mask=True, return_origin=True,
                                                  verbose=False)
    n_changes = sum(s_.count(PAL) for s_ in manip) - sum(s_.count(PAL) for s_ in ipa)
    tag = "".join(consonants) + "j"
    print(f"{len(ipa)} sentences, {len(words)} words, {n_changes} consonants palatalized ({', '.join(consonants)})")
    stats = block_changes(dict(ipa=ipa, manip=manip), label="Palatalized (B2, ʲ inside the consonant)")

    def st_of(v):
        c = {**base, **{k: v[k] for k in base if k in v}}
        if "weights" in v:
            c["WEIGHTS"] = v["weights"]
        return c, dict(timing=c["TIMING"], ramp_start=c["RAMP_START"], spill=c["SPILL_FRAMES"], speed=speed,
                       seed=seed, voice_name=_name(voice), context=c["CONTEXT"], cons_vector=c["CONS_VECTOR"],
                       prosody=c["PROSODY"])

    def describe(c, w):
        return (f"ʲ weight {w:g} (ramp {c['RAMP_START']:g}·w → w, fade {c['SPILL_FRAMES']} frame(s)) · "
                f"context {c['CONTEXT']} · consonant vector {c['CONS_VECTOR']} · prosody {c['PROSODY']} · "
                f"timing {c['TIMING']} · seed {seed}")

    sections, written = [], []
    _, st0 = st_of({})
    if make_reference:
        res = _render_text(ipa, manip, masks, pack, 0.0, st0, original=True)
        sections.append(("original", "Original (no changes)", _player_fragment(
            res, words, f"Original · voice {_name(voice)}", None, None, "original text")))
        sf.write(out / "b2_original.wav", res["audio"], SR); written.append(out / "b2_original.wav")
    if make_stock:
        res = _render_text(ipa, manip, masks, pack, 0.0, st0, stock=True)
        sections.append(("stock", "Stock Kokoro tʲ (ʲ has its own frames: extra j)", _player_fragment(
            res, None, f"Stock tʲ · voice {_name(voice)}", [rule], stats,
            "stock Kokoro: the ʲ token gets its own frames (word labels omitted: the string has extra tokens)")))
        sf.write(out / f"b2_{tag}_stock.wav", res["audio"], SR); written.append(out / f"b2_{tag}_stock.wav")

    for vi, v in enumerate(variants, start=1):
        c, st = st_of(v)
        for w in as_list(c["WEIGHTS"]):
            res = _render_text(ipa, manip, masks, pack, w, st)
            name = f"b2_{tag}_v{vi}_w{w:.2f}"
            head = f"V{vi} {v.get('label', '')} · ʲ weight {w:g}"
            sections.append((f"v{vi}_w{w:.2f}".replace(".", "_"), head, _player_fragment(
                res, words, f"{head} · voice {_name(voice)}", [rule], stats,
                describe(c, w) + " · dashed = original prosody")))
            sf.write(out / f"{name}.wav", res["audio"], SR); written.append(out / f"{name}.wav")
            print(f"V{vi} w={w:g}: {len(res['audio']) / SR:.1f} s   ({describe(c, w)})")

    page = out / f"b2_{tag}_palatal.html"
    title = f"B2 palatalization {' '.join(c_ + PAL for c_ in consonants)} · voice {_name(voice)}"
    intro = (f"{n_changes} consonants changed in {len(ipa)} sentences. "
             "V-numbers are the variants (see each player's description line). "
             "Phonemes marked as changed carry the ʲ; it has no frames of its own. "
             "Pitch panel: solid = this condition, dashed = original prosody.")
    page.write_text(_page(title, intro, sections), encoding="utf-8")
    written.insert(0, page)
    print(f"written to {out}/ :")
    for p_ in written:
        print("   ", Path(p_).name)
    return written


if __name__ == "__main__":
    run()

2 sentences, 23 words, 12 consonants palatalized (k)
V1 w=0.4: 8.0 s   (ʲ weight 0.4 (ramp 0.3·w → w, fade 1 frame(s)) · context original · consonant vector palatal · prosody original · timing original · seed 0)
V1 w=0.7: 8.0 s   (ʲ weight 0.7 (ramp 0.3·w → w, fade 1 frame(s)) · context original · consonant vector palatal · prosody original · timing original · seed 0)
V2 w=0: 8.0 s   (ʲ weight 0 (ramp 0.3·w → w, fade 1 frame(s)) · context original · consonant vector palatal · prosody original · timing original · seed 0)
V3 w=0.4: 8.0 s   (ʲ weight 0.4 (ramp 0.3·w → w, fade 1 frame(s)) · context palatal · consonant vector palatal · prosody original · timing original · seed 0)
V3 w=0.7: 8.0 s   (ʲ weight 0.7 (ramp 0.3·w → w, fade 1 frame(s)) · context palatal · consonant vector palatal · prosody original · timing original · seed 0)
V4 w=0.4: 8.0 s   (ʲ weight 0.4 (ramp 0.3·w → w, fade 2 frame(s)) · context palatal · consonant vector palatal · prosody blended · timing original · seed 0)
V